<img src="../../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

# 03 · Model interfaces and composition

**Goal:** Wrap a native PyTorch energy model, inspect a supported AIMNet2 adapter, and compose dependent and independent energy terms through public Toolkit interfaces.

**Core concepts:** `ModelConfig` declares capabilities and runtime choices. `BaseModelMixin` adapts native inputs and outputs. `PipelineStep`, `PipelineGroup`, and `PipelineModelWrapper` make data flow and derivative ownership explicit.

In [ ]:
import warnings

import helpers
import pandas as pd
import torch
from ase import Atoms, units
from IPython.display import HTML, display
from nvalchemi.data import AtomicData, Batch
from nvalchemi.models import (
    AIMNet2Wrapper,
    DFTD3ModelWrapper,
    PipelineGroup,
    PipelineModelWrapper,
    PipelineStep,
)
from nvalchemi.models.base import (
    BaseModelMixin,
    ModelConfig,
    NeighborConfig,
    NeighborListFormat,
)
from nvalchemi.neighbors import compute_neighbors

warnings.filterwarnings("ignore", message="Can't initialize NVML")
helpers.configure_presentation()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

<details>
<summary>Where NVIDIA ALCHEMI fits (recap)</summary>

[ALCHEMI](https://developer.nvidia.com/cuda/cuda-x-libraries/alchemi) provides Toolkit model and workflow interfaces, Toolkit-Ops kernels, and deployable NIM services. This lesson uses ALCHEMI Toolkit with its PyTorch data path and Toolkit-Ops neighbor, electrostatic, and dispersion operations.

- [Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/)
- [Toolkit-Ops documentation](https://nvidia.github.io/nvalchemi-toolkit-ops/)
- [Supported model wrappers](https://nvidia.github.io/nvalchemi-toolkit/models/index.html)

The notebook uses CUDA when available and CPU otherwise. On a CPU-only system, the collapsed setup keeps Warp's driver-discovery log outside the main lesson. Import or kernel failures still raise normally.
</details>

## Where this lesson fits

The [Core Playbook](../00-core-playbook/alchemi-core-playbook.ipynb) introduced model evaluation. Part 01 built [`AtomicData` and `Batch`](../01-atomicdata-batch/atomicdata-and-batch.ipynb), and Part 02 rebuilt batches from [stored records](../02-zarr-data-loading/zarr-data-loading.ipynb). Here those objects cross a reusable model boundary.

<object data="../../shared/curriculum-map-03.svg" type="image/svg+xml" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;" aria-label="ALCHEMI Toolkit course path with Part 03 model interfaces and composition highlighted.">
  <img src="../../shared/curriculum-map-03.svg" alt="ALCHEMI Toolkit course path from atomic data and loading through model interfaces, hooks, dynamics, and larger GPU workflows, with Part 03 highlighted." style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">
</object>

Part 03 adds one operation: treat a model as an inspectable, configurable, composable Toolkit object.

## Start with a model you can check by hand

`NativeQuadratic` assigns each atom the deterministic energy $s\lVert\mathbf{r}\rVert^2$ and sums atom energies by graph. The coordinates and energies use tutorial units. This synthetic energy only tests the interface; it has no chemical interpretation.

The wrapper will translate three native names, return Toolkit-standard energy and forces, and expose its choices through `ModelConfig`.

> **ALCHEMI Toolkit API**
> A concrete wrapper combines `torch.nn.Module` with `BaseModelMixin`, sets `model_config` in `__init__`, and implements the native input, output, and embedding boundaries.

In [ ]:
toy_atoms = [
    Atoms("H2", positions=[[0.0, 0.0, 0.0], [0.8, 0.0, 0.0]]),
    Atoms("H", positions=[[0.0, 0.4, 0.0]]),
]
toy_graphs = [
    AtomicData.from_atoms(atoms, device=device, dtype=torch.float64)
    for atoms in toy_atoms
]
toy_batch = Batch.from_data_list(toy_graphs, device=device)

In [ ]:
class NativeQuadratic(torch.nn.Module):
    """A native graph energy with deliberately non-Toolkit names."""

    def __init__(self, scale: float) -> None:
        super().__init__()
        self.scale = scale

    def forward(self, coordinates, graph_index, graph_count):
        atom_energy = self.scale * coordinates.square().sum(dim=1)
        energy = torch.zeros(
            graph_count,
            dtype=coordinates.dtype,
            device=coordinates.device,
        ).index_add(0, graph_index.long(), atom_energy)
        return {"native_energy": energy.unsqueeze(-1)}

In [ ]:
class QuadraticWrapper(torch.nn.Module, BaseModelMixin):
    """Expose NativeQuadratic through the Toolkit model interface."""

    def __init__(self, model: NativeQuadratic) -> None:
        super().__init__()
        self.model = model
        self.model_config = ModelConfig(
            outputs=frozenset({"energy", "forces"}),
            active_outputs={"energy", "forces"},
            autograd_outputs=frozenset({"forces"}),
            autograd_inputs=frozenset({"positions"}),
        )

    def adapt_input(self, data: Batch, **kwargs):
        model_inputs = super().adapt_input(data, **kwargs)
        return {
            "coordinates": model_inputs["positions"],
            "graph_index": data.batch_idx,
            "graph_count": data.num_graphs,
        }

    def adapt_output(self, raw_output, data: Batch):
        output = super().adapt_output(
            {"energy": raw_output["native_energy"]}, data
        )
        if "forces" in self.model_config.active_outputs:
            output["forces"] = -torch.autograd.grad(
                raw_output["native_energy"].sum(),
                data.positions,
                create_graph=self.training,
            )[0]
        return output

    @property
    def embedding_shapes(self) -> dict[str, tuple[int, ...]]:
        return {}

    def compute_embeddings(self, data, **kwargs):
        raise NotImplementedError("This analytical model has no embeddings.")

    def forward(self, data: Batch, **kwargs):
        native_inputs = self.adapt_input(data, **kwargs)
        return self.adapt_output(self.model(**native_inputs), data)

In [ ]:
toy_model = QuadraticWrapper(NativeQuadratic(scale=0.25)).to(device).eval()
{
    "available outputs": sorted(toy_model.model_config.outputs),
    "active outputs": sorted(toy_model.model_config.active_outputs),
    "Toolkit input": "positions",
    "native input": "coordinates",
    "native output": "native_energy",
    "Toolkit output": "energy",
}

In [ ]:
toy_native_positions = toy_batch.positions.detach().clone().requires_grad_(True)
toy_native_output = toy_model.model(
    toy_native_positions, toy_batch.batch_idx, toy_batch.num_graphs
)
toy_native_forces = -torch.autograd.grad(
    toy_native_output["native_energy"].sum(), toy_native_positions
)[0]
toy_output = toy_model(toy_batch)
torch.testing.assert_close(toy_output["energy"], toy_native_output["native_energy"])
torch.testing.assert_close(toy_output["forces"], toy_native_forces)
toy_energy_delta = (toy_output["energy"] - toy_native_output["native_energy"]).abs().max().detach().item()
toy_force_delta = (toy_output["forces"] - toy_native_forces).abs().max().detach().item()
toy_correction = QuadraticWrapper(NativeQuadratic(scale=0.10)).to(device).eval()
toy_sum = toy_model + toy_correction
toy_sum_output = toy_sum(toy_batch.clone())
toy_expected = toy_model(toy_batch.clone())["energy"] + toy_correction(
    toy_batch.clone()
)["energy"]
toy_closure = (toy_sum_output["energy"] - toy_expected).abs().max().detach().item()
assert toy_closure <= 1.0e-12
pd.DataFrame(
    {"maximum difference": [toy_energy_delta, toy_force_delta, toy_closure]},
    index=["native/wrapper energy", "native/wrapper force", "additive energy"],
)

The wrapper reproduces the native energy and force, and `toy_model + toy_correction` returns a `PipelineModelWrapper` with two independent direct groups. Each group reads the same `Batch`; neither group supplies an input to the other.

## Transfer the interface to a molecular calculation

The real example uses the supported `aimnet2-wb97m-d3_0` checkpoint on one phenol/N-methylacetamide AB/A/B triplet from the [NCI Atlas](https://github.com/Honza-R/NCIAtlas), distributed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The three finite graphs contain H, C, N, and O and have zero declared charge.

The checkpoint comes from [AIMNetCentral](https://github.com/isayevlab/aimnetcentral/) under its MIT license. Toolkit wraps the checked local file through the public `from_checkpoint(...)` constructor. The wrapper evaluates the checkpoint base with its built-in Coulomb and dispersion paths disabled, so we can compose those terms explicitly.

In [ ]:
aimnet = AIMNet2Wrapper.from_checkpoint(
    helpers.model_checkpoint(), device=device, compile_model=False
).eval()
aimnet = helpers.freeze_model(aimnet)

In [ ]:
records = helpers.load_nci_records()
nci_records = records[
    records["system_id"].eq("1.041") & records["scale"].eq(1.0)
].reset_index(drop=True)
assert nci_records["fragment"].tolist() == ["AB", "A", "B"]

In [ ]:
nci_atoms = [
    helpers.atoms_from_record(record)
    for record in nci_records.to_dict(orient="records")
]

In [ ]:
nci_graphs = [
    AtomicData.from_atoms(atoms, device=device, dtype=torch.float32)
    for atoms in nci_atoms
]
nci_batch = Batch.from_data_list(nci_graphs, device=device)

In [ ]:
input_summary = nci_records[
    ["fragment", "system_name", "interaction_class", "natoms", "charge"]
].copy()
input_summary["scope"] = ["finite, nonperiodic"] * len(input_summary)
input_summary.set_index("fragment")

## Inspect the supported adapter before calling it

Read the live `model_config` instead of assuming model behavior. AIMNet2 exposes available and active outputs, required and optional inputs, float32 parameters, device, periodic support, and a full MATRIX neighbor list at its declared cutoff.

Energy has shape `[B, 1]` in eV. Atomic charges have shape `[V]` in elementary-charge units. Forces, when requested, have shape `[V, 3]` in eV/Å. Interface support does not establish scientific suitability.

In [ ]:
aimnet_contract = helpers.model_contract_table(aimnet)

In [ ]:
aimnet_contract

In [ ]:
aimnet.set_config("active_outputs", {"energy", "charges"})
{
    "active outputs": sorted(aimnet.model_config.active_outputs),
    "device": str(next(aimnet.parameters()).device),
    "dtype": str(next(aimnet.parameters()).dtype),
}

### Prepare exactly the neighbors the adapter requests

For one evaluation, `compute_neighbors(...)` reads `aimnet.model_config.neighbor_config` and writes the declared full MATRIX representation onto the `Batch`. Iterative workflows use `make_neighbor_hooks()` so this preparation runs at the correct workflow stage.

In [ ]:
compute_neighbors(nci_batch, config=aimnet.model_config.neighbor_config)
{
    "neighbor matrix shape": tuple(nci_batch.neighbor_matrix.shape),
    "cutoff / Å": aimnet.model_config.neighbor_config.cutoff,
    "format": aimnet.model_config.neighbor_config.format.value,
    "full list": not aimnet.model_config.neighbor_config.half_list,
}

In [ ]:
aimnet_output = aimnet(nci_batch)

In [ ]:
aimnet_output_contract = helpers.output_contract_table(
    aimnet_output,
    num_graphs=nci_batch.num_graphs,
    num_nodes=nci_batch.num_nodes,
)

In [ ]:
aimnet_output_contract

### Check charge before using it downstream

AIMNet2 returns one predicted partial charge per atom. Their graph-wise sums must reproduce the declared AB, A, and B charges within `2e-6 e` before the Coulomb adapter consumes them.

In [ ]:
predicted_graph_charge = torch.zeros(
    nci_batch.num_graphs, device=device
).index_add(0, nci_batch.batch_idx.long(), aimnet_output["charges"])
expected_graph_charge = nci_batch.charge.reshape(-1).to(predicted_graph_charge)
charge_residual_e = float(
    (predicted_graph_charge - expected_graph_charge).abs().max()
)
assert charge_residual_e <= 2.0e-6

In [ ]:
pd.DataFrame(
    {
        "predicted sum / e": predicted_graph_charge.detach().cpu(),
        "declared charge / e": expected_graph_charge.detach().cpu(),
        "absolute residual / e": (
            predicted_graph_charge - expected_graph_charge
        ).abs().detach().cpu(),
    },
    index=nci_records["fragment"],
)

## Wrap the finite-system Coulomb term

The native model expects `neighbor_pairs`, `partial_charges`, and graph membership. Toolkit names the full COO list `neighbor_list`. `DirectCoulombAdapter` declares the requirement, maps those names, and derives forces from energy.

> **Highlight**
> The adapter owns interface translation. The native model keeps one job: evaluate its finite point-charge energy over a full neighbor list.

In [ ]:
class NativeCoulomb(torch.nn.Module):
    """Finite point-charge energy over a full COO neighbor list."""

    def forward(
        self, positions, partial_charges, neighbor_pairs, batch_idx, num_graphs
    ):
        source, target = neighbor_pairs.unbind(dim=1)
        distances = torch.linalg.vector_norm(
            positions[source] - positions[target], dim=-1
        )
        pair_energy = (
            0.5
            * units.Hartree
            * units.Bohr
            * partial_charges[source]
            * partial_charges[target]
            / distances
        )
        energy = torch.zeros(
            num_graphs, dtype=positions.dtype, device=positions.device
        ).index_add(0, batch_idx[source].long(), pair_energy)
        return {"native_energy": energy.unsqueeze(-1)}

In [ ]:
class DirectCoulombAdapter(torch.nn.Module, BaseModelMixin):
    """Expose NativeCoulomb through the Toolkit model interface."""

    def __init__(self, model: NativeCoulomb, cutoff: float) -> None:
        super().__init__()
        self.model = model
        self.model_config = ModelConfig(
            outputs=frozenset({"energy", "forces"}),
            active_outputs={"energy", "forces"},
            autograd_outputs=frozenset({"forces"}),
            autograd_inputs=frozenset({"positions"}),
            required_inputs=frozenset({"partial_charges"}),
            supports_pbc=False,
            needs_pbc=False,
            neighbor_config=NeighborConfig(
                cutoff=cutoff,
                format=NeighborListFormat.COO,
                half_list=False,
            ),
        )

    def adapt_input(self, data: Batch, **kwargs):
        model_inputs = super().adapt_input(data, **kwargs)
        return {
            "positions": model_inputs["positions"],
            "partial_charges": model_inputs["partial_charges"].reshape(-1),
            "neighbor_pairs": model_inputs["neighbor_list"],
            "batch_idx": data.batch_idx,
            "num_graphs": data.num_graphs,
        }

    def adapt_output(self, raw_output, data: Batch):
        output = super().adapt_output(
            {"energy": raw_output["native_energy"]}, data
        )
        if "forces" in self.model_config.active_outputs:
            output["forces"] = -torch.autograd.grad(
                raw_output["native_energy"].sum(),
                data.positions,
                create_graph=self.training,
            )[0]
        return output

    @property
    def embedding_shapes(self) -> dict[str, tuple[int, ...]]:
        return {}

    def compute_embeddings(self, data, **kwargs):
        raise NotImplementedError("This analytical model has no embeddings.")

    def forward(self, data: Batch, **kwargs):
        native_inputs = self.adapt_input(data, **kwargs)
        return self.adapt_output(self.model(**native_inputs), data)

In [ ]:
coulomb = DirectCoulombAdapter(
    NativeCoulomb(), cutoff=95.0 * units.Bohr
).to(device).eval()

In [ ]:
pd.DataFrame(
    [
        ("input", "positions", "positions"),
        ("input", "partial_charges", "partial_charges"),
        ("input", "neighbor_list", "neighbor_pairs"),
        ("output", "native_energy", "energy"),
    ],
    columns=["boundary", "native key", "Toolkit key"],
).set_index(["boundary", "native key"])

In [ ]:
helpers.model_contract_table(coulomb)

### Verify native-wrapper parity before composition

Both routes use the same NCI geometries, detached AIMNet2 charges, and full COO neighbor list. `Batch.add_key(...)` registers the per-atom `partial_charges` field through the public ownership-aware API.

In [ ]:
boundaries = nci_batch.batch_ptr.tolist()
charge_parts = [
    aimnet_output["charges"][boundaries[i] : boundaries[i + 1]]
    .detach().clone()
    for i in range(nci_batch.num_graphs)
]
native_batch = helpers.build_batch(nci_records, device=device)
native_batch.add_key("partial_charges", charge_parts, level="node")
native_batch.positions.requires_grad_(True)
compute_neighbors(native_batch, config=coulomb.model_config.neighbor_config)

In [ ]:
native_output = coulomb.model(
    positions=native_batch.positions,
    partial_charges=native_batch.partial_charges,
    neighbor_pairs=native_batch.neighbor_list,
    batch_idx=native_batch.batch_idx,
    num_graphs=native_batch.num_graphs,
)
native_forces = -torch.autograd.grad(
    native_output["native_energy"].sum(), native_batch.positions
)[0]

In [ ]:
adapter_batch = helpers.build_batch(nci_records, device=device)
adapter_batch.add_key("partial_charges", charge_parts, level="node")
compute_neighbors(adapter_batch, config=coulomb.model_config.neighbor_config)

In [ ]:
adapter_output = coulomb(adapter_batch)
torch.testing.assert_close(
    adapter_output["energy"], native_output["native_energy"], rtol=0, atol=1e-6
)
torch.testing.assert_close(
    adapter_output["forces"], native_forces, rtol=0, atol=1e-6
)

In [ ]:
pd.DataFrame(
    {
        "maximum absolute difference": [
            float(
                (adapter_output["energy"] - native_output["native_energy"])
                .abs().max()
            ),
            float((adapter_output["forces"] - native_forces).abs().max()),
        ],
        "unit": ["eV", "eV/Å"],
    },
    index=["energy", "force component"],
)

## Compose dependent and independent terms

AIMNet2 produces `charges`; the Coulomb adapter requires `partial_charges`. The explicit `PipelineStep` names that one translation. Their energies share autograd because Coulomb energy depends on positions directly and through AIMNet2's position-dependent charges.

D3(BJ) reads the same geometry independently and computes its own direct forces. The pinned environment creates and verifies its parameter tensors from the upstream Bonn table outside this repository. The notebook sets `auto_download=False` and does not redistribute that generated cache. See the [official DFT-D3 example](https://nvidia.github.io/nvalchemi-toolkit-ops/main/examples/dispersion/01_dftd3_molecule.html).

How do values move through the composed model?

```mermaid
%%{init: {"theme":"base","flowchart":{"curve":"linear"},"themeVariables":{"fontFamily":"NVIDIA Sans, Arial, sans-serif","background":"#000000","lineColor":"#7C8794"}}}%%
flowchart LR
  B("Batch") --> A("AIMNet2 base<br/>energy + charges")
  A -->|"charges to partial_charges"| C("finite Coulomb<br/>energy")
  A --> G("shared autograd group")
  C --> G
  B --> D("D3(BJ)<br/>energy + direct forces")
  G --> O("complete outputs")
  D --> O
  classDef active fill:#76B900,color:#050505,stroke:#76B900;
  classDef context fill:#20252B,color:#F3F4F6,stroke:#30363D;
  class A,C,G,O active;
  class B,D context;
```

AIMNet2 charges cross one named wire. D3 joins through a separate independent group.

In [ ]:
d3 = DFTD3ModelWrapper(
    a1=0.566,
    a2=3.128,
    s8=0.3908,
    s6=1.0,
    cutoff=95.0 * units.Bohr,
    smoothing_fraction=0.0,
    auto_download=False,
    param_file=helpers.d3_parameter_file(),
).to(device).eval()

In [ ]:
aimnet.set_config("active_outputs", {"energy", "forces", "charges"})
coulomb.set_config("active_outputs", {"energy", "forces"})
d3.set_config("active_outputs", {"energy", "forces"})

In [ ]:
charge_step = PipelineStep(
    aimnet, wire={"charges": "partial_charges"}
)
dependent_group = PipelineGroup(
    steps=[charge_step, coulomb], use_autograd=True
)
dispersion_group = PipelineGroup(steps=[d3], use_autograd=False)

In [ ]:
full_model = PipelineModelWrapper(
    groups=[dependent_group, dispersion_group],
    neighbor_adaptation="always",
).eval()
full_model.set_config("active_outputs", {"energy", "forces", "charges"})

In [ ]:
helpers.pipeline_table(full_model)

### Let the composed model describe neighbor preparation

The three components request different cutoffs and MATRIX or COO layouts. `neighbor_adaptation="always"` is a deliberate one-shot choice for this 50-atom finite batch: build one maximum-cutoff source and adapt it for each step. `full_model.make_neighbor_hooks()` exposes the equivalent preparation point for iterative workflows. Larger systems should inspect the default `"auto"` plan instead of assuming that one long list is efficient.

In [ ]:
full_neighbor_hooks = full_model.make_neighbor_hooks()
{
    "hook count": len(full_neighbor_hooks),
    "hook types": [type(hook).__name__ for hook in full_neighbor_hooks],
    "source cutoff / Å": full_model.model_config.neighbor_config.cutoff,
}

In [ ]:
full_batch = helpers.build_batch(nci_records, device=device)
compute_neighbors(
    full_batch, config=full_model.model_config.neighbor_config
)

In [ ]:
full_outputs = full_model(full_batch)
assert torch.isfinite(full_outputs["energy"]).all()
assert torch.isfinite(full_outputs["forces"]).all()

In [ ]:
helpers.output_contract_table(
    full_outputs,
    num_graphs=full_batch.num_graphs,
    num_nodes=full_batch.num_nodes,
)

## Check charge and component closure

Charge closure established that the predicted atomic charges match each graph's declared total. Component closure now asks a separate execution question: does the composed graph energy equal the AIMNet2 checkpoint base plus finite Coulomb plus D3(BJ), evaluated on the same AB/A/B geometries?

The force check stays at the native-wrapper boundary. Adding standalone AIMNet2 and frozen-charge Coulomb forces would omit the response of predicted charges to position. The shared-autograd group owns that derivative in the complete model.

In [ ]:
d3_batch = helpers.build_batch(nci_records, device=device)
compute_neighbors(d3_batch, config=d3.model_config.neighbor_config)
d3_output = d3(d3_batch)

In [ ]:
component_sum = (
    aimnet_output["energy"].detach()
    + adapter_output["energy"].detach()
    + d3_output["energy"].detach()
)
component_closure_eV = float(
    (full_outputs["energy"] - component_sum).abs().max()
)
assert component_closure_eV <= 2.0e-5

In [ ]:
pd.DataFrame(
    {
        "maximum absolute difference / eV": [component_closure_eV],
        "limit / eV": [2.0e-5],
        "passed": [component_closure_eV <= 2.0e-5],
    },
    index=["complete energy vs component sum"],
)

In [ ]:
fragments = nci_records["fragment"].tolist()
component_energies = {
    "AIMNet2 checkpoint base": aimnet_output["energy"].detach(),
    "finite Coulomb": adapter_output["energy"].detach(),
    "D3(BJ)": d3_output["energy"].detach(),
    "complete model": full_outputs["energy"].detach(),
}
component_interactions = pd.Series(
    {
        name: helpers.ab_minus_a_minus_b(energy, fragments).item()
        * helpers.EV_TO_KCAL_MOL
        for name, energy in component_energies.items()
    },
    name="interaction energy / kcal/mol",
)

In [ ]:
dft_reference = helpers.ab_minus_a_minus_b(
    nci_records["wb97m_d3bj_def2_tzvppd_total_energy_kcal_mol"].to_numpy(),
    fragments,
)
reference_kcal_mol = float(
    nci_records["ccsd_t_cbs_interaction_energy_kcal_mol"].iat[0]
)
pd.concat(
    [
        component_interactions,
        pd.Series(
            {
                "DFT-D3 reference at this geometry": dft_reference,
                "CCSD(T)/CBS reference at this geometry": reference_kcal_mol,
            },
            name=component_interactions.name,
        ),
    ]
).to_frame()

### What does each architectural term contribute to AB - A - B?

The bars below are architectural model terms. They explain how this configured calculation is assembled. They do not partition the electronic energy into uniquely defined physical observables.

The complete value and two references describe one fixed geometry. Agreement or disagreement at this point does not establish accuracy, transferability, or a validated interaction curve. These finite, nonperiodic graphs also do not support a condensed-phase claim.

In [ ]:
display(
    HTML(
        helpers.component_plot_html(
            component_interactions,
            reference_kcal_mol=reference_kcal_mol,
        )
    )
)

## Try it: narrow the requested output

Request only energy from the supplied AIMNet2 adapter, run the already prepared AB/A/B batch, and inspect the result. Success means the mapping contains only `energy` with shape `[3, 1]`.

In [ ]:
aimnet.set_config("active_outputs", {"energy"})
exercise_outputs = aimnet(nci_batch)
assert set(exercise_outputs) == {"energy"}
assert exercise_outputs["energy"].shape == (3, 1)
{
    "keys": set(exercise_outputs),
    "shape": tuple(exercise_outputs["energy"].shape),
}

## Recap

### What you can now do

- Inspect `model_config` for outputs, inputs, precision, device, periodic support, and neighbor requirements.
- Change runtime output selection with `set_config(...)` and inspect the next result.
- Wrap native code with `BaseModelMixin`, `ModelConfig`, `adapt_input`, and `adapt_output`.
- Use `model_a + model_b` for independent additive terms.
- Use `PipelineStep`, `PipelineGroup`, and `PipelineModelWrapper` for named dependencies and shared autograd.
- Check native-wrapper parity, graph charge, output contracts, and component closure before interpreting a composition.

### Where these interfaces go next

- [Hooks](../04-hooks/hooks.ipynb) registers model-provided neighbor hooks with observation and control hooks.
- [BaseDynamics](../05-base-dynamics/base-dynamics.ipynb) passes the same configured model object into optimization and molecular dynamics.
- [Training and fine-tuning](../07-training-finetuning/training-finetuning.ipynb) is in progress and will reuse `ModelConfig` for trainable models.

Go deeper with the official [model-wrapping guide](https://nvidia.github.io/nvalchemi-toolkit/userguide/models.html), [supported-model table](https://nvidia.github.io/nvalchemi-toolkit/models/index.html), [additive LJ + Ewald example](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/07_composable_model_composition.html), and [AIMNet2 + Ewald pipeline example](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/08_aimnet2_ewald_pipeline.html).